In [1]:
import pandas as pd

left = pd.DataFrame({
    "id": [1, 1, 2],
    "value": [10, 11, 20]
})

right = pd.DataFrame({
    "id": [1, 1, 2],
    "name": ["A", "A_dup", "B"]
})

merged = left.merge(right, on="id", how="left")

print("Строк слева:", len(left))
print("Строк после merge:", len(merged))
display(merged)

Строк слева: 3
Строк после merge: 5


,id,value,name
0,1,10,A
1,1,10,A_dup
2,1,11,A
3,1,11,A_dup
4,2,20,B


In [2]:
right_unique = (
    right
    .drop_duplicates(subset=["id"], keep="first")
)

merged_safe = left.merge(
    right_unique,
    on="id",
    how="left",
    validate="many_to_one"
)

print("Строк после безопасного merge:", len(merged_safe))
display(merged_safe)

Строк после безопасного merge: 3


,id,value,name
0,1,10,A
1,1,11,A
2,2,20,B


In [3]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\IDD23\OneDrive\Desktop\е\end-to-end-project"
)

NORMALIZED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weather_hourly_normalized.csv"
)

df = pd.read_csv(NORMALIZED_PATH)

df["time"] = pd.to_datetime(df["time"])

print("Размер normalized:", df.shape)
print("Колонки:", df.columns.tolist())
display(df.head())

Размер normalized: (168, 5)
Колонки: ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m']


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,2026-09-15 00:00:00,13.1,79,0.0,7.6
1,2026-09-15 01:00:00,12.4,82,0.0,6.1
2,2026-09-15 02:00:00,11.8,82,0.0,5.8
3,2026-09-15 03:00:00,11.4,83,0.0,5.0
4,2026-09-15 04:00:00,11.0,83,0.0,4.7


In [4]:
df["city_id"] = "variant_02_location"

In [5]:
reference = pd.DataFrame({
    "city_id": ["variant_02_location"],
    "city_name": ["Исходная локация"],
    "source_type": ["open-meteo"],
    "timezone": ["UTC"]
})

print("Строк normalized до join:", len(df))
print("Уникальных city_id в reference:", reference["city_id"].nunique())
print("Дубликаты ключей reference:", reference["city_id"].duplicated().sum())

Строк normalized до join: 168
Уникальных city_id в reference: 1
Дубликаты ключей reference: 0


In [6]:
enriched = df.merge(
    reference,
    on="city_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Строк после join:", len(enriched))
print("Результат соединения:")
print(enriched["_merge"].value_counts())

display(enriched.head())

Строк после join: 168
Результат соединения:
_merge
both          168
left_only       0
right_only      0
Name: count, dtype: int64


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,city_id,city_name,source_type,timezone,_merge
0,2026-09-15 00:00:00,13.1,79,0.0,7.6,variant_02_location,Исходная локация,open-meteo,UTC,both
1,2026-09-15 01:00:00,12.4,82,0.0,6.1,variant_02_location,Исходная локация,open-meteo,UTC,both
2,2026-09-15 02:00:00,11.8,82,0.0,5.8,variant_02_location,Исходная локация,open-meteo,UTC,both
3,2026-09-15 03:00:00,11.4,83,0.0,5.0,variant_02_location,Исходная локация,open-meteo,UTC,both
4,2026-09-15 04:00:00,11.0,83,0.0,4.7,variant_02_location,Исходная локация,open-meteo,UTC,both


In [7]:
mart_daily = (
    enriched
    .set_index("time")
    .resample("D")
    .agg(
        city_name=("city_name", "first"),
        source_type=("source_type", "first"),
        temperature_avg=("temperature_2m", "mean"),
        temperature_min=("temperature_2m", "min"),
        temperature_max=("temperature_2m", "max"),
        precipitation_total=("precipitation", "sum"),
        humidity_avg=("relative_humidity_2m", "mean"),
        wind_speed_avg=("wind_speed_10m", "mean"),
        wind_speed_max=("wind_speed_10m", "max"),
        observation_count=("temperature_2m", "count")
    )
    .reset_index()
)

numeric_columns = [
    "temperature_avg",
    "temperature_min",
    "temperature_max",
    "precipitation_total",
    "humidity_avg",
    "wind_speed_avg",
    "wind_speed_max"
]

mart_daily[numeric_columns] = mart_daily[numeric_columns].round(2)

display(mart_daily)

,time,city_name,source_type,temperature_avg,temperature_min,temperature_max,precipitation_total,humidity_avg,wind_speed_avg,wind_speed_max,observation_count
0,2026-09-15,Исходная локация,open-meteo,13.81,10.8,16.5,0.00,66.29,5.98,10.4,24
1,2026-09-16,Исходная локация,open-meteo,14.03,11.8,17.0,0.10,78.00,12.69,18.0,24
2,2026-09-17,Исходная локация,open-meteo,15.38,13.3,18.1,3.75,81.71,15.27,19.4,24
3,2026-09-18,Исходная локация,open-meteo,13.73,11.1,16.7,0.90,79.88,14.65,18.4,24
4,2026-09-19,Исходная локация,open-meteo,14.79,12.7,17.5,2.10,74.04,16.60,23.7,24
5,2026-09-20,Исходная локация,open-meteo,14.77,11.7,18.6,0.20,76.04,15.04,18.9,24
6,2026-09-21,Исходная локация,open-meteo,14.05,10.2,17.3,3.40,81.42,11.78,17.5,24


In [8]:
mart_daily["temperature_avg_3d_rolling"] = (
    mart_daily["temperature_avg"]
    .rolling(window=3, min_periods=1)
    .mean()
    .round(2)
)

mart_daily["precipitation_3d_rolling"] = (
    mart_daily["precipitation_total"]
    .rolling(window=3, min_periods=1)
    .sum()
    .round(2)
)

display(
    mart_daily[
        [
            "time",
            "temperature_avg",
            "temperature_avg_3d_rolling",
            "precipitation_total",
            "precipitation_3d_rolling"
        ]
    ]
)

,time,temperature_avg,temperature_avg_3d_rolling,precipitation_total,precipitation_3d_rolling
0,2026-09-15,13.81,13.81,0.00,0.00
1,2026-09-16,14.03,13.92,0.10,0.10
2,2026-09-17,15.38,14.41,3.75,3.85
3,2026-09-18,13.73,14.38,0.90,4.75
4,2026-09-19,14.79,14.63,2.10,6.75
5,2026-09-20,14.77,14.43,0.20,3.20
6,2026-09-21,14.05,14.54,3.40,5.70


In [9]:
MART_DIR = (
    PROJECT_ROOT
    / "data"
    / "mart"
    / "variant_02"
)

MART_DIR.mkdir(parents=True, exist_ok=True)

MART_PATH = MART_DIR / "mart_daily_weather.csv"

mart_daily.to_csv(
    MART_PATH,
    index=False,
    encoding="utf-8"
)

print("Mart сохранён:")
print(MART_PATH)
print("Размер mart:", mart_daily.shape)

Mart сохранён:
C:\Users\IDD23\OneDrive\Desktop\е\end-to-end-project\data\mart\variant_02\mart_daily_weather.csv
Размер mart: (7, 13)
